# 03b · IMERG for the three Andean potato sites (2024)

**Why.** The rotation scoring in notebook 04 depends on the rainfall that NASA POWER assigns to each site. In Huancayo and Puno POWER gives ~400 mm/year, which looks low for potato zones. IMERG (satellite rainfall, 0.1°) is an independent source: if it also gives low totals, the limitation is real; if it gives much more, POWER is the problem at those points (as already seen in Ipiales and Nairobi, notebook 03).

**What it does.** Same extraction as notebook 03 (same functions, same variables), only for Ventaquemada, Huancayo and Puno, 2024. Resumable; ~9 s per day, about 55 min in total. Output: `data/imerg_final_daily_andean.csv`. At the end, a monthly POWER vs IMERG comparison per site.

Run it in a separate kernel while notebook 04 keeps waiting for AppEEARS; the two do not interfere.

In [ ]:
import sys, os, re, time, getpass, warnings
from pathlib import Path
sys.path.append("../src")
import numpy as np, pandas as pd
import climate_tools as ct
warnings.filterwarnings("ignore")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 80)
import earthaccess, xarray as xr
from concurrent.futures import ThreadPoolExecutor

SITES = {"Ventaquemada": (5.37, -73.52, 2618), "Huancayo": (-12.07, -75.21, 3248), "Puno": (-15.84, -70.02, 3823)}
DATA_DIRS = ["../data", "data"]
OUT = Path("../data"); OUT.mkdir(exist_ok=True)
IMERG_CSV = OUT / "imerg_final_daily_andean.csv"

def show(obj, title=None, digits=2):
    if title: print("\n" + title + "\n" + "-" * len(title))
    ff = lambda v: (f"{v:.2e}" if (v != 0 and abs(v) < 1e-3) else f"{v:.{digits}f}")
    print(obj.to_string(float_format=ff) if isinstance(obj, (pd.DataFrame, pd.Series)) else obj)

if not os.environ.get("EARTHDATA_TOKEN"):
    os.environ["EARTHDATA_TOKEN"] = getpass.getpass("Earthdata token (input hidden): ").strip()
auth = earthaccess.login(strategy="environment")
print("authenticated:", bool(auth.authenticated))

In [ ]:
def open_ds(fileobj):
    try:
        ds = xr.open_dataset(fileobj, engine="h5netcdf")
        if "precipitation" not in ds: raise KeyError("precipitation")
    except KeyError:
        ds = xr.open_dataset(fileobj, engine="h5netcdf", group="Grid")
    return ds

def spatial_vars(ds):
    return [v for v in ds.data_vars if {"lat", "lon"} <= set(ds[v].dims) <= {"time", "lat", "lon"}]

def date_of(granule):
    return pd.to_datetime(re.search(r"3IMERG\.(\d{8})", granule.data_links()[0]).group(1))

def block(ds, var, i0, j0, half):
    da = ds[var]
    if "time" in da.dims: da = da.isel(time=0)
    a, b = max(i0 - half, 0), max(j0 - half, 0)
    da = da.isel(lat=slice(a, i0 + half + 1), lon=slice(b, j0 + half + 1)).transpose("lat", "lon")
    return np.asarray(da.values, float), i0 - a, j0 - b

def extract_file(fileobj, sites=SITES, half=5):
    ds = open_ds(fileobj); li, lo_i = ds.indexes["lat"], ds.indexes["lon"]
    others = [v for v in spatial_vars(ds) if v != "precipitation"]
    rows = []
    for n, (la, lo, el) in sites.items():
        i0 = int(li.get_indexer([la], method="nearest")[0]); j0 = int(lo_i.get_indexer([lo], method="nearest")[0])
        b, ci, cj = block(ds, "precipitation", i0, j0, half)
        w5 = b[max(ci - 2, 0):ci + 3, max(cj - 2, 0):cj + 3]
        r = {"site": n, "cell_lat": float(li[i0]), "cell_lon": float(lo_i[j0]),
             "P_center": b[ci, cj], "P_mean5": np.nanmean(w5), "P_sd5": np.nanstd(w5), "P_max5": np.nanmax(w5),
             "P_wet5": float(np.nanmean(w5 >= 1)), "P_mean11": np.nanmean(b), "P_sd11": np.nanstd(b)}
        for v in others:
            bb, c1, c2 = block(ds, v, i0, j0, 0); r[v] = bb[0, 0]
        rows.append(r)
    ds.close(); return pd.DataFrame(rows)

START, END = "2024-01-01", "2024-12-31"
BATCH = 30
done = set(pd.read_csv(IMERG_CSV, usecols=["date"])["date"].unique()) if IMERG_CSV.exists() else set()
granules = earthaccess.search_data(short_name="GPM_3IMERGDF", version="07", temporal=(START, END))
todo = [g for g in granules if date_of(g).strftime("%Y-%m-%d") not in done]
todo.sort(key=date_of, reverse=True)
print(f"granules in window: {len(granules)} | already stored: {len(granules) - len(todo)} | to download: {len(todo)}")

def process(args):
    g, f = args
    d = date_of(g).strftime("%Y-%m-%d")
    try:
        df = extract_file(f); df.insert(0, "date", d); return df
    except Exception as e:
        return ("ERROR", d, repr(e)[:200])

errors, t0 = [], time.time()
for i in range(0, len(todo), BATCH):
    chunk = todo[i:i + BATCH]
    files = earthaccess.open(chunk)
    with ThreadPoolExecutor(max_workers=2) as ex:
        res = list(ex.map(process, zip(chunk, files)))
    good = [r for r in res if isinstance(r, pd.DataFrame)]
    errors += [r for r in res if not isinstance(r, pd.DataFrame)]
    if good:
        new = pd.concat(good)
        if IMERG_CSV.exists():
            cols = pd.read_csv(IMERG_CSV, nrows=0).columns; new = new.reindex(columns=cols)
        new.to_csv(IMERG_CSV, mode="a", header=not IMERG_CSV.exists(), index=False)
    print(f"{i + len(chunk)}/{len(todo)} days | {time.time() - t0:.0f} s elapsed | errors so far: {len(errors)}")
print("finished. errors:", errors[:5])

## POWER vs IMERG at the three Andean sites (2024)

Same reading as in notebook 03: monthly totals from the two sources, the ratio of annual totals, and the Spearman correlation of the monthly series (does the timing agree even when the totals do not?). The 5x5 pixel mean (~50 km) is also shown because POWER is itself a ~50 km grid.

In [ ]:
im = pd.read_csv(IMERG_CSV, parse_dates=["date"]).drop_duplicates(["date", "site"]).sort_values(["site", "date"])
print("days per site:", im.groupby("site").date.nunique().to_dict())
rows, monthly = {}, []
for n, (la, lo, el) in SITES.items():
    pw = ct.load_csv(ct.find_csv(la, lo, DATA_DIRS))["PRECTOTCORR"].loc["2024"]
    d = im[im.site == n].set_index("date")
    m = pd.DataFrame({"POWER_mm": pw.resample("MS").sum(), "IMERG_pixel_mm": d["P_center"].resample("MS").sum(min_count=20),
                      "IMERG_mean5_mm": d["P_mean5"].resample("MS").sum(min_count=20)}).dropna()
    m.insert(0, "site", n); monthly.append(m)
    rows[n] = {"POWER_mm_yr": pw.sum(), "IMERG_pixel_mm_yr": d["P_center"].sum(), "IMERG_mean5_mm_yr": d["P_mean5"].sum(),
               "ratio_pixel_over_POWER": d["P_center"].sum() / pw.sum(),
               "monthly_rank_corr": m["POWER_mm"].corr(m["IMERG_pixel_mm"], method="spearman"),
               "wet_days_POWER(>=1mm)": int((pw >= 1).sum()), "wet_days_IMERG(>=1mm)": int((d["P_center"] >= 1).sum())}
MON = pd.concat(monthly); MON.index.name = "month"
show(pd.DataFrame(rows).T, "Annual totals and agreement, 2024", 2)
for n in SITES: show(MON[MON.site == n].drop(columns="site").round(1), f"Monthly rain, {n}")
MON.round(2).to_csv(OUT / "platform_rain_monthly_andean.csv"); pd.DataFrame(rows).T.round(3).to_csv(OUT / "platform_rain_sites_andean.csv")
print("saved: platform_rain_monthly_andean.csv, platform_rain_sites_andean.csv")